In [ ]:
# ==========================================
# TESIS ZMVM: PM, clima y salud
# Autor: Arely Leal
# Descripción: Script en Python para generar grafico de tendencia a la mortalida para la ZMVM. 
# Periodo: 2000-2019
# ==========================================

In [ ]:
#GRAFICAR DE FORMA INDIVIDUAL

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# === RUTAS DE ENTRADA ===
archivo_poblacion = "RUTA DEL ARCHIVO"
ruta_salida = "RUTA DE SALIDA"
os.makedirs(ruta_salida, exist_ok=True)

ruta_defunciones = {
    "respiratorias": "RUTA DEL ARCHIVO MORTALIDAD",
    "cardiovasculares": "RUTA DEL ARCHIVO MORTALIDAD",
    "metabólicas": "RUTA DEL ARCHIVO MORTALIDAD"
}

# === CARGA DE POBLACIÓN ===
df_pob = pd.read_csv(archivo_poblacion, encoding='latin1')
df_pob = df_pob.rename(columns={'CLAVE': 'CVEGEO'})

# === FUNCIÓN PARA CALCULAR TASA ANUAL ZMVM ===
def calcular_tasa_zmvm(archivo_defunciones):
    df = pd.read_csv(archivo_defunciones, encoding='latin1')
    df = df[df['Anio'].between(2000, 2019)]

    poblacion_total = df_pob.groupby(['CVEGEO', 'Anio'])['POB_TOTAL'].sum().reset_index()
    defunciones_total = df.groupby(['CVEGEO', 'Anio']).size().reset_index(name='Defunciones')

    df_tasa = defunciones_total.merge(poblacion_total, on=['CVEGEO', 'Anio'], how='left')

    resumen = df_tasa.groupby('Anio').agg({
        'Defunciones': 'sum',
        'POB_TOTAL': 'sum'
    }).reset_index()

    resumen['Tasa_mortalidad'] = (resumen['Defunciones'] / resumen['POB_TOTAL']) * 10000
    return resumen

# === CALCULAR TODAS LAS SERIES Y LA ESCALA GLOBAL ===
series = {}
todas_tasas = []

for nombre, ruta in ruta_defunciones.items():
    resumen = calcular_tasa_zmvm(ruta)
    series[nombre] = resumen
    todas_tasas.extend(resumen['Tasa_mortalidad'].tolist())

y_min = 0
y_max = max(todas_tasas)
y_max = round(y_max + 0.5)

print(f"Escala comparable: {y_min} a {y_max}")

# === GRAFICAR POR SEPARADO CON MISMA ESCALA ===
for nombre_enfermedad, resumen in series.items():
    plt.figure(figsize=(10, 6))

    sns.regplot(
        x='Anio',
        y='Tasa_mortalidad',
        data=resumen,
        ci=95,
        line_kws={"color": "dimgray", "linewidth": 2},
        scatter_kws={"s": 40, "color": "black", "zorder": 3},
        color='lightgray'
    )

    plt.suptitle(
        f"MORTALIDAD POR ENFERMEDADES {nombre_enfermedad.upper()}",
        fontsize=14, fontweight='bold', y=0.96
    )
    plt.title("Zona Metropolitana del Valle de México", fontsize=12, y=1.02)
    plt.xlabel("Año", fontweight='bold', labelpad=15, fontsize=10)
    plt.ylabel("Mortalidad (por cada diez mil habitantes)", fontweight='bold', labelpad=15, fontsize=10)

    plt.ylim(y_min, y_max)
    plt.xticks(range(2000, 2020, 1))
    plt.xticks(rotation=0, fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.grid(True, linestyle='--', alpha=0.2)
    plt.tight_layout()

    salida = os.path.join(ruta_salida, f"tendencia_zmvm_{nombre_enfermedad.lower()}.png")
    plt.savefig(salida, dpi=300)
    plt.close()

    print(f" Gráfico guardado: {salida}")

Escala comparable: 0 a 12
 Gráfico guardado: /Users/arelyleal/Downloads/TESIS/BASES DE DATOS/MORTALIDAD/TENDENCIA_COMPARABLE/tendencia_zmvm_respiratorias.png
 Gráfico guardado: /Users/arelyleal/Downloads/TESIS/BASES DE DATOS/MORTALIDAD/TENDENCIA_COMPARABLE/tendencia_zmvm_cardiovasculares.png
 Gráfico guardado: /Users/arelyleal/Downloads/TESIS/BASES DE DATOS/MORTALIDAD/TENDENCIA_COMPARABLE/tendencia_zmvm_metabólicas.png


In [ ]:
#GRAFICA COMPARATIVA PARA LOS TRES GRUPOS DE ENFERMEDADES

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# === RUTAS DE ENTRADA ===
archivo_poblacion = "RUTA DEL ARCHIVO"
ruta_salida = "RUTA DE SALIDA"
os.makedirs(ruta_salida, exist_ok=True)

ruta_defunciones = {
    "Respiratorias": "RUTA DEL ARCHIVO",
    "Cardiovasculares": "RUTA DEL ARCHIVO",
    "Metabólicas": "RUTA DEL ARCHIVO"
}

# === CARGA DE POBLACIÓN ===
df_pob = pd.read_csv(archivo_poblacion, encoding='latin1')
df_pob = df_pob.rename(columns={'CLAVE': 'CVEGEO'})

# === FUNCIÓN PARA CALCULAR TASA ANUAL ZMVM ===
def calcular_tasa_zmvm(archivo_defunciones):
    df = pd.read_csv(archivo_defunciones, encoding='latin1')
    df = df[df['Anio'].between(2000, 2019)]

    poblacion_total = df_pob.groupby(['CVEGEO', 'Anio'])['POB_TOTAL'].sum().reset_index()
    defunciones_total = df.groupby(['CVEGEO', 'Anio']).size().reset_index(name='Defunciones')

    df_tasa = defunciones_total.merge(poblacion_total, on=['CVEGEO', 'Anio'], how='left')

    resumen = df_tasa.groupby('Anio').agg({
        'Defunciones': 'sum',
        'POB_TOTAL': 'sum'
    }).reset_index()

    resumen['Tasa_mortalidad'] = (resumen['Defunciones'] / resumen['POB_TOTAL']) * 10000
    return resumen

# === CALCULAR SERIES ===
series = {}
todas_tasas = []

for nombre, ruta in ruta_defunciones.items():
    resumen = calcular_tasa_zmvm(ruta)
    series[nombre] = resumen
    todas_tasas.extend(resumen['Tasa_mortalidad'].tolist())

y_min = 0
y_max = max(todas_tasas)
y_max = round(y_max + 0.5)

# === GRÁFICA CONJUNTA ===
plt.figure(figsize=(10, 6))

colores = {
    "Respiratorias": "#a6cee3",   # azul pastel
    "Cardiovasculares": "#fdbf6f", # naranja pastel
    "Metabólicas": "#b2df8a"      # verde pastel
}
for nombre, resumen in series.items():
    plt.plot(
        resumen["Anio"],
        resumen["Tasa_mortalidad"],
        label=nombre,
        color=colores[nombre],
        linewidth=1.5,
        marker="o",
        markersize=4
    )

fig = plt.gcf()

fig.suptitle(
    "TENDENCIA MORTALIDAD",
    fontsize=14,
    fontweight='bold',
    y=0.987
)

fig.text(
    0.5, 0.92,   
    "ZMVM",
    ha='center',
    fontsize=12
)
plt.xlabel("Año", fontweight='bold', labelpad=15, fontsize=10)
plt.ylabel("Mortalidad (por cada diez mil habitantes)", fontweight='bold', labelpad=15, fontsize=10)

plt.ylim(y_min, y_max)
plt.xticks(range(2000, 2020, 1), fontsize=8)
plt.yticks(fontsize=8)
plt.grid(True, linestyle='--', alpha=0.2)
plt.legend(frameon=False, loc="best")
plt.tight_layout()

salida = os.path.join(ruta_salida, "tendencia_zmvm_comparativa.png")
plt.savefig(salida, dpi=300)
plt.close()

print(f"Gráfico comparativo guardado en: {salida}")

Gráfico comparativo guardado en: /Users/arelyleal/Downloads/TESIS/BASES DE DATOS/MORTALIDAD/TENDENCIA_COMPARABLE/tendencia_zmvm_comparativa.png
